# Summary

This notebook grants a service principal the minimum access needed to work with Unity Catalog objects in the `cosmos_poc` catalog. It:

- grants `USE CATALOG` on `cosmos_poc`
- grants schema access on `raw`, `ops`, and `bronze`
- grants read/write access on the required volumes
- grants `READ` access to the `kv-poc` secret scope
- verifies the resulting Unity Catalog permissions

In [0]:
APP_ID = "c6dda43a-2db4-42d5-a4ec-2d09231620b7"

grants = [
    f"GRANT USE CATALOG ON CATALOG cosmos_poc TO `{APP_ID}`",
    f"GRANT USE SCHEMA ON SCHEMA cosmos_poc.raw TO `{APP_ID}`",
    f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME cosmos_poc.raw.cosmos_landing TO `{APP_ID}`",
    f"GRANT USE SCHEMA ON SCHEMA cosmos_poc.ops TO `{APP_ID}`",
    f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME cosmos_poc.ops.checkpoints TO `{APP_ID}`",
    f"GRANT USE SCHEMA, CREATE TABLE ON SCHEMA cosmos_poc.bronze TO `{APP_ID}`",
]
for g in grants:
    spark.sql(g)
    print("OK:", g.replace(APP_ID, "<APP_ID>"))

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import AclPermission

w = WorkspaceClient()
w.secrets.put_acl(scope="kv-poc", principal=APP_ID, permission=AclPermission.READ)
print([(a.principal[-4:], a.permission.value) for a in w.secrets.list_acls(scope="kv-poc")])

In [0]:
display(spark.sql(f"SHOW GRANTS `{APP_ID}` ON VOLUME cosmos_poc.raw.cosmos_landing"))
display(spark.sql(f"SHOW GRANTS `{APP_ID}` ON SCHEMA cosmos_poc.bronze"))

In [0]:
spark.sql(f"GRANT CREATE TABLE, MODIFY, SELECT ON SCHEMA cosmos_poc.ops TO `{APP_ID}`")
# old layout from the first version - remove so nothing stale remains
dbutils.fs.rm("/Volumes/cosmos_poc/ops/checkpoints/extract_customers", True)
dbutils.fs.rm("/Volumes/cosmos_poc/raw/cosmos_landing/customers", True)
print("Grant applied, old checkpoint and landing folder removed")